# Is a drug-combination surface really a quadratic? A test on 192 measured surfaces

QPOP's core assumption: response to a drug combination is a smooth quadratic surface, so a few measured points (none / low / high dose of each drug) are enough to predict the rest.
Here that assumption meets real data: **192 fully measured 8x8 dose-response matrices** (two drugs, 8 concentrations each, 64 measurements, % viability) in 10 breast cell lines (9 cancer lines and the non-tumorigenic MCF-10A),
from the open repository of DECREASE (Ianevski et al., *Nat Mach Intell* 2019, doi 10.1038/s42256-019-0122-4; data `210_Novel_Anticancer_combinations/192_Combinations_.xlsx`). Get it with `sh scripts/fetch_open_data.sh`.

For each matrix:
- **QPOP-style:** measure a 3x3 sub-grid (concentration index 0, 4, 7), fit the 2-drug quadratic (6 terms), predict the 55 points not measured.
- **Bliss baseline:** predict every combination from the two measured single-drug curves only (independent action), the textbook null model.
- **Best possible quadratic:** fit to all 64 points; how well can the shape fit at all?

Dose is coded by concentration index (concentrations are roughly log-spaced), as QPOP codes levels. Cell lines, not patients.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

raw = pd.read_excel("../ext/decrease/210_Novel_Anticancer_combinations/192_Combinations_.xlsx")
MATS = []
for k, g in raw.groupby("PairIndex"):
    c1, c2 = np.sort(g.Conc1.unique()), np.sort(g.Conc2.unique())
    if len(c1) == 8 and len(c2) == 8:
        V = g.pivot_table(index="Conc1", columns="Conc2", values="Response").loc[c1, c2].to_numpy() / 100
        MATS.append(dict(k=k, d1=g.Drug1.iloc[0], d2=g.Drug2.iloc[0], cell=g.Cell.iloc[0], V=V))
print("complete 8x8 matrices:", len(MATS), "| cell lines:", sorted({m["cell"] for m in MATS}))

I, J = np.meshgrid(np.arange(8), np.arange(8), indexing="ij")
def F(i, j):
    i, j = np.asarray(i, float), np.asarray(j, float)
    return np.stack([np.ones_like(i), i, j, i * j, i * i, j * j], -1)
FULL = F(I.ravel(), J.ravel())
INNER = np.zeros((8, 8), bool); INNER[1:, 1:] = True            # true combinations (both drugs present)

def quad9(V, levels=(0, 4, 7)):
    sub = [(a, b) for a in levels for b in levels]
    b = np.linalg.lstsq(F([a for a, _ in sub], [c for _, c in sub]), np.array([V[a, c] for a, c in sub]), rcond=None)[0]
    held = INNER.copy()
    for a, c in sub:
        held[a, c] = False
    return (FULL @ b).reshape(8, 8), held

def bliss(V):
    return V[0, 0] * np.outer(V[:, 0] / V[0, 0], V[0, :] / V[0, 0])

rows = []
for m in MATS:
    V = m["V"]; y = V.ravel()
    ball = np.linalg.lstsq(FULL, y, rcond=None)[0]
    r2 = 1 - ((y - FULL @ ball) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    P9, held = quad9(V); PB = bliss(V)
    best = V[INNER].min()
    rows.append(dict(r2_full=r2,
                     rmse_quad9=np.sqrt(((P9 - V)[held] ** 2).mean()),
                     rmse_bliss=np.sqrt(((PB - V)[held] ** 2).mean()),
                     regret_quad9=V[INNER][P9[INNER].argmin()] - best,
                     regret_bliss=V[INNER][PB[INNER].argmin()] - best))
RES = pd.DataFrame(rows)
print(RES.median().round(3).to_string())
print("share of matrices where Bliss predicts the unmeasured combinations better:", round((RES.rmse_bliss < RES.rmse_quad9).mean(), 2))

complete 8x8 matrices: 192 | cell lines: ['BT-549', 'CAL-85-1', 'CAL148', 'CAL51', 'DU4475', 'HCC1599', 'MCF-10A', 'MDAMB-436', 'MDAMB231', 'MFM-223']
r2_full         0.898
rmse_quad9      0.106
rmse_bliss      0.074
regret_quad9    0.000
regret_bliss    0.000
share of matrices where Bliss predicts the unmeasured combinations better: 0.54


## 1. How well can a quadratic fit a real surface, even with all 64 points?

In [2]:
fig = go.Figure(go.Histogram(x=RES.r2_full, xbins=dict(start=0, end=1, size=0.025)))
fig.add_vline(x=RES.r2_full.median(), line_dash="dot", annotation_text=f"median {RES.r2_full.median():.2f}")
fig.update_layout(xaxis_title="R² of quadratic fitted to all 64 measurements", yaxis_title="matrices", height=380)
fig

## 2. QPOP-style 9 points vs the Bliss baseline: error on the unmeasured combinations

In [3]:
lim = float(max(RES.rmse_quad9.max(), RES.rmse_bliss.max())) * 1.05
fig = go.Figure(go.Scatter(x=RES.rmse_bliss, y=RES.rmse_quad9, mode="markers", opacity=0.6,
                           text=[f"{m['d1']} + {m['d2']} ({m['cell']})" for m in MATS]))
fig.add_trace(go.Scatter(x=[0, lim], y=[0, lim], mode="lines", line=dict(dash="dot"), showlegend=False))
fig.update_layout(xaxis_title="RMSE, Bliss from single-drug curves", yaxis_title="RMSE, quadratic from 3x3 points",
                  height=480, showlegend=False, title="Points above the line: Bliss was more accurate")
fig

## 3. Does the answer depend on which three dose levels are measured?

In [4]:
LEVELS = [(0, 2, 5), (0, 3, 6), (0, 4, 7), (0, 5, 7), (0, 3, 7)]
lev = {}
for L in LEVELS:
    e = []
    for m in MATS:
        P, held = quad9(m["V"], L)
        e.append(np.sqrt(((P - m["V"])[held] ** 2).mean()))
    lev[str(L)] = float(np.median(e))
print({k: round(v, 3) for k, v in lev.items()}, "| Bliss:", round(float(RES.rmse_bliss.median()), 3))
fig = go.Figure(go.Bar(x=list(lev), y=list(lev.values())))
fig.add_hline(y=float(RES.rmse_bliss.median()), line_dash="dot", annotation_text="Bliss baseline")
fig.update_layout(xaxis_title="concentration indices measured", yaxis_title="median RMSE on unmeasured combinations", height=380)
fig

{'(0, 2, 5)': 0.186, '(0, 3, 6)': 0.115, '(0, 4, 7)': 0.106, '(0, 5, 7)': 0.124, '(0, 3, 7)': 0.11} | Bliss: 0.074


## 4. Real surfaces next to their QPOP-style fit (rotate; pick a matrix)

In [5]:
order = RES.r2_full.sort_values().index
picks = [order[int(q * (len(order) - 1))] for q in (0.1, 0.5, 0.9)]   # poor, typical, good quadratic fit
fig = go.Figure()
for n, i in enumerate(picks):
    V = MATS[i]["V"]; P, _ = quad9(V)
    fig.add_trace(go.Surface(z=V, colorscale="Greys", opacity=0.55, showscale=False, visible=n == 0, name="measured"))
    fig.add_trace(go.Surface(z=P, colorscale="Viridis", opacity=0.75, showscale=False, visible=n == 0, name="quadratic from 3x3"))
btn = [dict(label=f"{MATS[i]['d1']} + {MATS[i]['d2']} ({MATS[i]['cell']}), R² {RES.r2_full[i]:.2f}", method="update",
            args=[dict(visible=[j // 2 == n for j in range(2 * len(picks))])]) for n, i in enumerate(picks)]
fig.update_layout(updatemenus=[dict(buttons=btn, x=0, y=1.12, xanchor="left")], height=620,
                  scene=dict(xaxis_title="drug 2 index", yaxis_title="drug 1 index", zaxis_title="viability"),
                  title="Grey: measured. Colour: quadratic from the 9 QPOP-style points")
fig

## What this says

1. On real measured surfaces the quadratic is a rough shape: median R² about 0.90 even with all 64 points, and about half the matrices fit worse than that.
2. Fitted QPOP-style from 9 points, it predicts the unmeasured combinations **less accurately than the plain Bliss baseline** built from the two single-drug curves (median RMSE about 0.10 vs 0.07), and the error depends strongly on which dose levels are measured.
3. For picking the most effective dose pair, the two are close (see the regret columns above); neither needs QPOP's interaction model to do that well on these data.
4. This is in cell lines with one measurement per point. It does not test QPOP's 12-drug designs, and it cannot test clinical value. It does test QPOP's stated foundation, that responses are "smooth parabolic surfaces", and that foundation holds only roughly.

Building on existing work: the data and the idea of predicting full matrices from few measurements come from DECREASE, which fits a matrix-factorisation + boosting model for the same task. Running DECREASE itself (R) against QPOP's quadratic on these matrices is the natural next step.